# Demo guiada — una función de Churn y una interfaz Streamlit

Este caso sintético pertenece a los primeros 60 minutos de la clase 1 y prepara la Práctica 5.1. No usa el bundle Wine ni sustituye la práctica de clase 2. El objetivo es observar cómo Streamlit envuelve una función `predict()` ya probada.

La regla es determinista y visible: su score **no es una probabilidad calibrada**. S5 se limita a función, widgets, formulario, rerun y salida; estado, caché, FSM y telemetría se reservan para S6.

In [ ]:
from pathlib import Path
import sys

relative_solution = Path('semana5/modules/05-streamlit-basic-model-ui/solutions/01-churn-streamlit')
solution_root = next(
    parent / relative_solution
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / relative_solution).is_dir()
)
for import_path in (solution_root, solution_root / 'src'):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

import app as churn_app
from churn_demo.model import predict

print('Solución docente:', solution_root)
print('Función:', predict.__module__ + '.' + predict.__name__)

## 1. Leer el contrato antes de la pantalla

| Entrada | Tipo | Rango |
| --- | --- | ---: |
| `tenure_months` | entero | 0–120 |
| `monthly_spend_eur` | número | 0–300 |
| `support_calls` | entero | 0–20 |
| `has_annual_contract` | booleano | sí/no |

La salida contiene `will_churn`, `label`, `risk_score` y `explanation`. La interfaz no decide el score.

### Predice antes de ejecutar

1. ¿Cuál tendrá mayor score: `2, 95, 4, False` o `36, 35, 0, True`?
2. ¿Qué factores empujan cada decisión?
3. ¿Puede el score salir del intervalo 0,05–0,95?
4. ¿Por qué no debemos llamarlo probabilidad?

In [ ]:
profiles = {
    'riesgo_alto': (2, 95, 4, False),
    'riesgo_bajo': (36, 35, 0, True),
    'inicial': (12, 60, 1, False),
}

results = {name: predict(*values) for name, values in profiles.items()}
for name, result in results.items():
    print(f"{name}: {result['label']} — {result['risk_score']:.2f}")
    print(result['explanation'])

In [ ]:
assert results['riesgo_alto']['risk_score'] == 0.95
assert results['riesgo_bajo']['risk_score'] == 0.05
assert results['inicial']['risk_score'] == 0.45
assert set(results['inicial']) == {
    'will_churn', 'label', 'risk_score', 'explanation'
}
print('Contrato determinista comprobado sin interfaz.')

## 2. Modelo mental de Streamlit

```text
rerun → dibujar formulario → ¿submitted?
                              ├─ no: 0 llamadas y una instrucción
                              └─ sí: 1 llamada → resultado o error
```

`st.form` agrupa los cambios. La inferencia se coloca detrás de `st.form_submit_button`; no se llama a la función por cada ajuste. En S5 aceptamos el rerun y no añadimos mecanismos de estado o caché.

In [ ]:
assert churn_app.predict is predict
print('La interfaz importa la misma función que acabamos de probar:', True)
print('La regla vive en:', Path(predict.__code__.co_filename).name)

## 3. Preparar la Práctica 5.1

El starter conserva `model.py` completo y deja tres responsabilidades en `app.py`:

| Responsabilidad | Evidencia |
| --- | --- |
| construir cuatro widgets dentro de un único formulario | claves y componentes correctos |
| llamar a `predict(**values)` solo después del submit | cero llamadas al editar y una al enviar |
| presentar etiqueta, score, explicación o error seguro | salida visible sin detalle interno |

Estas tareas ocupan los últimos 60 minutos de la clase. Los tests convierten cada responsabilidad en un comportamiento observable.

## 4. Transferir el patrón, no los datos ni el código

La clase 2 vuelve a Wine Quality y utiliza el bundle real de S4. Conserva el formulario, el submit, la semántica cero/una llamada y la presentación segura. Cambia a once campos, `InferenceGateway.predict(values)` y `PredictionPayload`.

```text
Churn: 4 widgets → predict() → salida explicable
Wine: 11 widgets → gateway → PredictionPayload
```

No se copia ningún archivo de Churn a la práctica Wine. Pregunta de cierre: si una app Wine y una CLI Wine discrepan para la misma muestra, ¿qué frontera comprobarías primero?